In [2]:
import sys
sys.path.insert(0, '/data/batch-jobs')


In [3]:
import torch

torch.load("/data/training_data/cellxgene_v2_training_v1_shuffled_genept_1536/batch_0000.pt")

/tmp/ipykernel_99646/2808019535.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("/data/training_data/cellxgene_v2_training_v1_shuffled_genept_1536/batch_0000.

{'X': tensor([[-0.0072,  0.0141, -0.0068,  ..., -0.0205,  0.0016, -0.0017],
         [-0.0044,  0.0157, -0.0058,  ..., -0.0201,  0.0025,  0.0013],
         [-0.0083,  0.0112, -0.0042,  ..., -0.0156,  0.0051, -0.0013],
         ...,
         [-0.0077,  0.0146, -0.0069,  ..., -0.0188,  0.0013, -0.0039],
         [-0.0066,  0.0130, -0.0066,  ..., -0.0187,  0.0024, -0.0005],
         [-0.0089,  0.0110, -0.0040,  ..., -0.0145,  0.0052, -0.0020]]),
 'row_hash': tensor([-2463678338245236318,  7444083912343952464,  1815046616382894859,
          ...,  3165209882513018331,  -253254267025955169,
          3150835203151554438]),
 'start_idx': 0,
 'end_idx': 10000,
 'n_samples': 10000}

In [21]:
import duckdb
file_uuid = "486486d4-9462-43e5-9249-eb43fa5a49a6"
file_path = f"/mmc-scratch/scratch/cellxgene_v2_training_v1/{file_uuid}.parquet"
test_cells_pdf = duckdb.sql(f"SELECT * FROM '{file_path}' LIMIT 5").df()
test_cells_pdf

,cell_type,assay,organism,0,1,2,3,4,5,6,...,suspension_type,HTAN_Biospecimen_ID,HTAN_Participant_ID,tissue_type,disease,sex,tissue,self_reported_ethnicity,development_stage,observation_joinid
0,regulatory T cell,10x 3' v2,Homo sapiens,-0.007439,0.008613,-0.003005,0.014610,-0.004533,-0.029216,0.042863,...,cell,HTA8_1017_1,HTA8_1017,tissue,lung adenocarcinoma,female,lung,European,65-year-old stage,1y-I6Oz=G@
1,regulatory T cell,10x 3' v3,Homo sapiens,-0.007726,0.012330,-0.003796,0.013425,-0.003667,-0.024397,0.042148,...,cell,HTA8_2015_1,HTA8_2015,tissue,small cell lung carcinoma,male,axilla,European,55-year-old stage,V09G{(DKM*
2,regulatory T cell,10x 3' v3,Homo sapiens,-0.009001,0.009638,-0.003720,0.015976,-0.006698,-0.028458,0.043036,...,cell,HTA8_2019_1,HTA8_2019,tissue,small cell lung carcinoma,male,liver,European,81-year-old stage,1ij9GeXT#T
3,regulatory T cell,10x 3' v3,Homo sapiens,-0.007244,0.010813,-0.003453,0.013749,-0.002610,-0.026385,0.042846,...,cell,HTA8_2010_1,HTA8_2010,tissue,small cell lung carcinoma,male,lung,European,62-year-old stage,?J=$gaO{X=
4,regulatory T cell,10x 3' v3,Homo sapiens,-0.008807,0.011097,-0.004259,0.013172,-0.005467,-0.025804,0.042269,...,cell,HTA8_2015_1,HTA8_2015,tissue,small cell lung carcinoma,male,axilla,European,55-year-old stage,OsVwuU9?Q~


In [22]:
from core.embeddings.shuffle import create_composite_ids, compute_row_hash
composite_ids = create_composite_ids(
  file_uuid,
  test_cells_pdf.observation_joinid.values
)
row_hashes = compute_row_hash(composite_ids)

In [23]:
row_hashes

array([ -524842275315172619, -1102403935128083820,  8979528685851105789,
       -3028119761974282334, -4508329130049779669])

In [24]:
import torch
from pathlib import Path

def find_row_in_batches(row_hash, batch_dir):
    """
    Find which batch contains the given row_hash and its position.
    
    Args:
        row_hash: int64 hash computed from composite_id
        batch_dir: Path to directory containing batch_*.pt files
    
    Returns:
        (batch_file, local_index) or (None, None) if not found
    """
    batch_dir = Path(batch_dir)

    for batch_file in sorted(batch_dir.glob("batch_*.pt")):
        batch_data = torch.load(batch_file, weights_only=False)
        row_hashes = batch_data["row_hash"]

        # Find if this hash exists in this batch
        matches = (row_hashes == row_hash).nonzero(as_tuple=True)[0]

        if len(matches) > 0:
            local_idx = matches[0].item()
            return batch_file, local_idx

    return None, None

In [ ]:
for row_hash in row_hashes:
  batch_file, idx = find_row_in_batches(
    row_hash,
    "/localdata/training_data/cellxgene_v2_training_v1_shuffled_metadata/"
  )

  if batch_file is not None:
    print(f"Found in {batch_file.name} at index {idx}")

    # Load and extract the specific row
    batch_data = torch.load(batch_file, weights_only=False)
    embedding = batch_data["X"][idx]  # Your embedding vector

    print(f"Embedding shape: {embedding.shape}")
    print(f"{embedding}")
  else:
    print("Not found in any batch")

Found in batch_0261.pt at index 7266
Embedding shape: (17,)
['regulatory T cell' 'CL:0000815' "10x 3' v2" 'EFO:0009899'
 'lung adenocarcinoma' 'MONDO:0005061' 'lung' 'UBERON:0002048'
 '65-year-old stage' 'HsapDv:0000159' 'European' 'HANCESTRO:0005' 'female'
 'PATO:0000383' 'Homo sapiens' 'NCBITaxon:9606' '1y-I6Oz=G@']
Found in batch_0065.pt at index 3603
Embedding shape: (17,)
['regulatory T cell' 'CL:0000815' "10x 3' v3" 'EFO:0009922'
 'small cell lung carcinoma' 'MONDO:0008433' 'axilla' 'UBERON:0009472'
 '55-year-old stage' 'HsapDv:0000149' 'European' 'HANCESTRO:0005' 'male'
 'PATO:0000384' 'Homo sapiens' 'NCBITaxon:9606' 'V09G{(DKM*']
Found in batch_0080.pt at index 3509
Embedding shape: (17,)
['regulatory T cell' 'CL:0000815' "10x 3' v3" 'EFO:0009922'
 'small cell lung carcinoma' 'MONDO:0008433' 'liver' 'UBERON:0002107'
 '81-year-old stage' 'HsapDv:0000207' 'European' 'HANCESTRO:0005' 'male'
 'PATO:0000384' 'Homo sapiens' 'NCBITaxon:9606' '1ij9GeXT#T']
Found in batch_0070.pt at ind

In [ ]:
for row_hash in row_hashes:
  batch_file, idx = find_row_in_batches(
    row_hash,
    "/localdata/training_data/cellxgene_v2_training_v1_shuffled_genept_1536/"
  )

  if batch_file is not None:
    print(f"Found in {batch_file.name} at index {idx}")

    # Load and extract the specific row
    batch_data = torch.load(batch_file, weights_only=False)
    embedding = batch_data["X"][idx]  # Your embedding vector

    print(f"Embedding shape: {embedding.shape}")
    print(f"{embedding}")
  else:
    print("Not found in any batch")

Found in batch_0261.pt at index 7266
Embedding shape: torch.Size([1536])
tensor([-0.0074,  0.0086, -0.0030,  ..., -0.0157,  0.0075, -0.0008])
Found in batch_0065.pt at index 3603
Embedding shape: torch.Size([1536])
tensor([-0.0077,  0.0123, -0.0038,  ..., -0.0161,  0.0052, -0.0014])
Found in batch_0080.pt at index 3509
Embedding shape: torch.Size([1536])
tensor([-0.0090,  0.0096, -0.0037,  ..., -0.0142,  0.0054, -0.0016])
Found in batch_0070.pt at index 4943
Embedding shape: torch.Size([1536])
tensor([-0.0072,  0.0108, -0.0035,  ..., -0.0163,  0.0061, -0.0011])
Found in batch_0155.pt at index 6681
Embedding shape: torch.Size([1536])
tensor([-0.0088,  0.0111, -0.0043,  ..., -0.0147,  0.0048, -0.0013])


In [25]:

import torch
import glob
import numpy as np

# Find training metadata files
metadata_dir = "/localdata/training_data/cellxgene_v2_training_v1_shuffled_metadata"
metadata_files = sorted(glob.glob(f"{metadata_dir}/batch_*.pt"))

print(f"Found {len(metadata_files)} metadata batch files")
print(f"First few files: {metadata_files[:3]}")

# Load first batch to examine labels
batch_data = torch.load(metadata_files[0])
print(f"\nFirst batch keys: {batch_data.keys()}")

# Examine the y values (labels)
y = batch_data['y']
print(f"\nLabel tensor shape: {y.shape}")
print(f"Label dtype: {y.dtype}")
print(f"Label range: min={y.min()}, max={y.max()}")
print(f"Unique labels: {torch.unique(y).tolist()}")
print(f"Number of unique labels: {len(torch.unique(y))}")

# Sample some labels
print(f"\nFirst 20 labels: {y[:20].tolist()}")


Found 377 metadata batch files
First few files: ['/localdata/training_data/cellxgene_v2_training_v1_shuffled_metadata/batch_0000.pt', '/localdata/training_data/cellxgene_v2_training_v1_shuffled_metadata/batch_0001.pt', '/localdata/training_data/cellxgene_v2_training_v1_shuffled_metadata/batch_0002.pt']

First batch keys: dict_keys(['X', 'row_hash', 'start_idx', 'end_idx', 'n_samples', 'y'])

Label tensor shape: torch.Size([10000])
Label dtype: torch.int64
Label range: min=0, max=547
Unique labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 1

/tmp/ipykernel_99646/577097174.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  batch_data = torch.load(metadata_files[0])


In [6]:

import pandas as pd
import glob

# Load validation data
val_genept_dir = "/localdata/training_data/cellxgene_v2_test_v1"
val_files = sorted(glob.glob(f"{val_genept_dir}/*.parquet"))

print(f"Found {len(val_files)} validation parquet files")
print(f"First file: {val_files[0]}")

# Load first validation file
df = pd.read_parquet(val_files[0])
print(f"\nValidation dataframe shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Check if y column exists and examine it
if 'y' in df.columns:
    print(f"\n'y' column dtype: {df['y'].dtype}")
    print(f"'y' range: min={df['y'].min()}, max={df['y'].max()}")
    print(f"Number of unique y values: {df['y'].nunique()}")
    print(f"First 20 y values: {df['y'].head(20).tolist()}")
else:
    print("\nNo 'y' column found!")
    print(f"Available columns: {df.columns.tolist()}")


Found 15 validation parquet files
First file: /localdata/training_data/cellxgene_v2_test_v1/05a49baa-d326-42ae-86d2-94de3a659901.parquet

Validation dataframe shape: (3770, 3108)
Columns: ['cell_type', 'assay', 'organism', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126'

In [7]:

# Look at cell_type values in validation data
print("Cell type values in validation parquet:")
print(f"dtype: {df['cell_type'].dtype}")
print(f"Number of unique: {df['cell_type'].nunique()}")
print(f"First 10 cell types: {df['cell_type'].head(10).tolist()}")
print(f"\nAre they strings? {isinstance(df['cell_type'].iloc[0], str)}")


Cell type values in validation parquet:
dtype: object
Number of unique: 9
First 10 cell types: ['fibroblast', 'fibroblast', 'fibroblast', 'fibroblast', 'fibroblast', 'fibroblast', 'fibroblast', 'fibroblast', 'fibroblast', 'fibroblast']

Are they strings? True


In [ ]:

# Now let's check what cell_type_codes is being used
# First, load one batch from training to see what cell types are being used

import sys
sys.path.insert(0, '/data/GenePT-tools/src')

from pathlib import Path
from data_loading.composable_dataset import ComposableTrainingDataset

# Create training dataset to see what cell types it has
train_dataset = ComposableTrainingDataset(
  base_dir=Path('/localdata/training_data'),
  embedding_types=['genept', 'scgpt', 'metadata'],
  batch_size=128,
  genept_dims=1536,
  seed=42,
  is_test_mode=False,
  start_batch_file=0,
  end_batch_file=1,  # Just load one batch
  verbose=False
)

print(f"Training dataset n_classes: {train_dataset.n_classes}")
print(f"Training dataset has cell_type_codes: {hasattr(train_dataset, 'cell_type_codes')}")

# Load metadata to get cell types
metadata_path = Path('/localdata/training_data/cellxgene_v2_training_v1_shuffled_metadata/metadata.pt')
metadata = torch.load(metadata_path, weights_only=True)

print(f"\nTraining metadata keys: {metadata.keys()}")
if 'cell_types' in metadata:
    cell_types = metadata['cell_types']
    print(f"Number of cell types in training: {len(cell_types)}")
    print(f"First 10 cell types: {cell_types[:10]}")



Training dataset n_classes: 549
Training dataset has cell_type_codes: True

Training metadata keys: dict_keys(['embedding_type', 'columns', 'n_total_samples', 'total_samples', 'n_batches', 'n_dims', 'batch_size', 'num_buckets', 'source_files', 'shuffle_seed', 'cell_types', 'created_at', 'version'])
Number of cell types in training: 549
First 10 cell types: ['fallopian tube secretory epithelial cell', 'perivascular cell', 'ciliated epithelial cell', 'stromal cell', 'lymphocyte', 'smooth muscle cell', 'endothelial cell of lymphatic vessel', 'plasma cell', 'mast cell', 'macrophage']


In [9]:

# Create the cell_type_codes mapping (cell_type -> numeric code)
cell_types_list = metadata['cell_types']
cell_type_codes = {ct: i for i, ct in enumerate(cell_types_list)}

print(f"Cell type codes dictionary size: {len(cell_type_codes)}")
print(f"\nFirst 5 mappings:")
for i, (ct, code) in enumerate(list(cell_type_codes.items())[:5]):
    print(f"  '{ct}' -> {code}")

# Check if validation cell types are in the training cell types
val_cell_types_in_file = df['cell_type'].unique()
print(f"\nCell types in validation file: {len(val_cell_types_in_file)}")
print(f"Validation cell types: {val_cell_types_in_file.tolist()}")

print(f"\nChecking if validation cell types are in training:")
for vct in val_cell_types_in_file:
    if vct in cell_type_codes:
        print(f"  '{vct}' -> {cell_type_codes[vct]} ✓")
    else:
        print(f"  '{vct}' -> NOT FOUND ✗")


Cell type codes dictionary size: 549

First 5 mappings:
  'fallopian tube secretory epithelial cell' -> 0
  'perivascular cell' -> 1
  'ciliated epithelial cell' -> 2
  'stromal cell' -> 3
  'lymphocyte' -> 4

Cell types in validation file: 9
Validation cell types: ['fibroblast', 'macrophage', 'T cell', 'B cell', 'malignant cell', 'endothelial cell', 'monocyte', 'mature NK T cell', 'hepatocyte']

Checking if validation cell types are in training:
  'fibroblast' -> 67 ✓
  'macrophage' -> 9 ✓
  'T cell' -> 274 ✓
  'B cell' -> 13 ✓
  'malignant cell' -> 289 ✓
  'endothelial cell' -> 10 ✓
  'monocyte' -> 71 ✓
  'mature NK T cell' -> 83 ✓
  'hepatocyte' -> 184 ✓


In [11]:

# Check what directories exist for validation
import os
base_dir = Path('/localdata/training_data')

# List all cellxgene_v2_test* directories
test_dirs = sorted([d for d in base_dir.iterdir() if d.is_dir() and 'test' in d.name])
print("Test directories:")
for d in test_dirs:
    file_count = len(list(d.glob('*.parquet')))
    print(f"  {d.name}: {file_count} parquet files")

# Check what columns are in the scgpt test directory
scgpt_test_dir = base_dir / 'cellxgene_v2_test_v1_scgpt'
if scgpt_test_dir.exists():
    first_file = sorted(scgpt_test_dir.glob('*.parquet'))[0]
    df_scgpt = pd.read_parquet(first_file)
    print(f"\nColumns in {scgpt_test_dir.name}/{first_file.name}:")
    print(f"  Total columns: {len(df_scgpt.columns)}")
    print(f"  First 10 columns: {df_scgpt.columns[:10].tolist()}")
    
    # Check for emb_ columns
    emb_cols = [col for col in df_scgpt.columns if col.startswith('emb_')]
    print(f"  Embedding columns (emb_*): {len(emb_cols)}")
    if emb_cols:
        print(f"  First 5 emb columns: {emb_cols[:5]}")


Test directories:
  cellxgene_v2_test_v1: 15 parquet files
  cellxgene_v2_test_v1_scgpt: 15 parquet files
  cellxgene_v2_test_v1_tissue: 15 parquet files

Columns in cellxgene_v2_test_v1_scgpt/05a49baa-d326-42ae-86d2-94de3a659901.parquet:
  Total columns: 513
  First 10 columns: ['observation_joinid', 'emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4', 'emb_5', 'emb_6', 'emb_7', 'emb_8']
  Embedding columns (emb_*): 512
  First 5 emb columns: ['emb_0', 'emb_1', 'emb_2', 'emb_3', 'emb_4']


In [12]:

# Try again with correct suffixes
test_dataset = ComposableTrainingDataset(
    base_dir=Path('/localdata/training_data'),
    embedding_types=['genept', 'scgpt', 'metadata'],
    batch_size=1024,
    genept_dims=1536,
    seed=42,
    is_test_mode=True,
    test_genept_suffix='_test_v1_scgpt',  # Correct suffix
    test_metadata_suffix='_test_v1',  
    test_tissue_suffix='_test_v1_tissue',
    cell_type_codes=cell_type_codes,
    verbose=False
)

print(f"Test dataset created successfully")
print(f"Number of files: {len(test_dataset.file_list)}")

# Load first batch and check labels
for X_batch, y_batch in test_dataset:
    print(f"\nFirst batch loaded:")
    print(f"X shape: {X_batch.shape}")
    print(f"y shape: {y_batch.shape}")
    print(f"y dtype: {y_batch.dtype}")
    print(f"y range: min={y_batch.min()}, max={y_batch.max()}")
    print(f"y unique values (first 20): {torch.unique(y_batch)[:20].tolist()}")
    
    # Decode labels back to cell type names
    reverse_mapping = {v: k for k, v in cell_type_codes.items()}
    print(f"\nFirst 20 labels decoded:")
    for i in range(min(20, len(y_batch))):
        code = y_batch[i].item()
        if code >= 0:
            cell_type = reverse_mapping.get(code, f"UNKNOWN({code})")
            print(f"  y[{i}] = {code:3d} -> '{cell_type}'")
        else:
            print(f"  y[{i}] = {code:3d} -> FILTERED")
    
    break  # Only check first batch


Test dataset created successfully
Number of files: 15

First batch loaded:
X shape: torch.Size([1024, 2048])
y shape: torch.Size([1024])
y dtype: torch.int64
y range: min=9, max=274
y unique values (first 20): [9, 67, 274]

First 20 labels decoded:
  y[0] =  67 -> 'fibroblast'
  y[1] =  67 -> 'fibroblast'
  y[2] =  67 -> 'fibroblast'
  y[3] =  67 -> 'fibroblast'
  y[4] =  67 -> 'fibroblast'
  y[5] =  67 -> 'fibroblast'
  y[6] =  67 -> 'fibroblast'
  y[7] =  67 -> 'fibroblast'
  y[8] =  67 -> 'fibroblast'
  y[9] =  67 -> 'fibroblast'
  y[10] =  67 -> 'fibroblast'
  y[11] =  67 -> 'fibroblast'
  y[12] =  67 -> 'fibroblast'
  y[13] =  67 -> 'fibroblast'
  y[14] =  67 -> 'fibroblast'
  y[15] =  67 -> 'fibroblast'
  y[16] =  67 -> 'fibroblast'
  y[17] =  67 -> 'fibroblast'
  y[18] =  67 -> 'fibroblast'
  y[19] =  67 -> 'fibroblast'


In [13]:

# Now let's check what the training labels look like vs validation labels
# to see if there's an offset or remapping issue

print("TRAINING LABELS:")
print(f"  Label range: 0 to {max(cell_type_codes.values())}")
print(f"  Number of classes: {len(cell_type_codes)}")
print(f"  Sample mappings:")
for i, (ct, code) in enumerate(list(cell_type_codes.items())[:10]):
    print(f"    {code:3d} -> '{ct}'")

print("\nVALIDATION LABELS (from test dataset):")
print(f"  Labels seen in first batch: {torch.unique(y_batch).tolist()}")
print(f"  Decoded:")
for code in torch.unique(y_batch).tolist():
    print(f"    {code:3d} -> '{reverse_mapping[code]}'")

# Now let's check what the model is predicting
# We need to load a trained model and run inference
print("\n" + "="*60)
print("Checking model predictions...")
print("="*60)

# Check if there's a checkpoint we can load
import glob
checkpoint_dir = Path('/localdata/checkpoints/mlp_composable_test')
if checkpoint_dir.exists():
    checkpoints = sorted(checkpoint_dir.glob('*.pth'))
    if checkpoints:
        latest_ckpt = checkpoints[-1]
        print(f"\nLatest checkpoint: {latest_ckpt.name}")
        
        # Load checkpoint
        checkpoint = torch.load(latest_ckpt, weights_only=False)
        print(f"Checkpoint keys: {checkpoint.keys()}")
        
        # Check what epoch this is
        if 'epoch' in checkpoint:
            print(f"Epoch: {checkpoint['epoch']}")
        if 'step' in checkpoint:
            print(f"Step: {checkpoint['step']}")


TRAINING LABELS:
  Label range: 0 to 548
  Number of classes: 549
  Sample mappings:
      0 -> 'fallopian tube secretory epithelial cell'
      1 -> 'perivascular cell'
      2 -> 'ciliated epithelial cell'
      3 -> 'stromal cell'
      4 -> 'lymphocyte'
      5 -> 'smooth muscle cell'
      6 -> 'endothelial cell of lymphatic vessel'
      7 -> 'plasma cell'
      8 -> 'mast cell'
      9 -> 'macrophage'

VALIDATION LABELS (from test dataset):
  Labels seen in first batch: [9, 67, 274]
  Decoded:
      9 -> 'macrophage'
     67 -> 'fibroblast'
    274 -> 'T cell'

Checking model predictions...


In [14]:

# Let's check what code_remapping is in the actual training run
# We need to look at where the trainer is instantiated

# First, let's see what parameters are passed when creating the trainer
# by looking at the train script

with open('/data/GenePT-tools/src/training/train.py', 'r') as f:
    train_script = f.read()

# Find where trainer is instantiated
import re
trainer_init = re.search(r'MLPTrainer\([^)]+\)', train_script, re.DOTALL)
if trainer_init:
    print("Trainer instantiation:")
    print(trainer_init.group(0)[:500])


FileNotFoundError: [Errno 2] No such file or directory: '/data/GenePT-tools/src/training/train.py'

In [15]:

# The issue is:
# 1. Training uses cell_type_codes that maps strings to ORIGINAL codes (0-548)
# 2. Then code_remapping filters these to NEW codes (0-N)
# 3. But validation also uses the SAME cell_type_codes (0-548)
# 4. And code_remapping should remap them to (0-N)

# The question is: is the validation dataset being passed code_remapping?
# Let's check line 286 in trainer.py again

print("Key insight:")
print("=" * 60)
print("Training flow:")
print("  1. cell_type_codes: {'fibroblast': 67, 'macrophage': 9, ...}")  
print("  2. code_remapping: {67: 42, 9: 5, ...} (original -> filtered)")
print("  3. During training: labels go 67 -> 42, 9 -> 5, etc.")
print()
print("Validation flow:")
print("  1. cell_type_codes: same as training {'fibroblast': 67, ...}")
print("  2. code_remapping: should be same {67: 42, 9: 5, ...}")
print("  3. During validation: labels should also go 67 -> 42, 9 -> 5")
print()
print("If code_remapping is NOT passed to validation dataset,")
print("then validation labels stay at 67, 9, etc.")
print("but model expects 42, 5, etc.")
print("=" * 60)

# Let's trace what's actually happening in the trainer
# by reading line 286


Key insight:
Training flow:
  1. cell_type_codes: {'fibroblast': 67, 'macrophage': 9, ...}
  2. code_remapping: {67: 42, 9: 5, ...} (original -> filtered)
  3. During training: labels go 67 -> 42, 9 -> 5, etc.

Validation flow:
  1. cell_type_codes: same as training {'fibroblast': 67, ...}
  2. code_remapping: should be same {67: 42, 9: 5, ...}
  3. During validation: labels should also go 67 -> 42, 9 -> 5

If code_remapping is NOT passed to validation dataset,
then validation labels stay at 67, 9, etc.
but model expects 42, 5, etc.


In [16]:

# The confusion is:
# When filtering is enabled (lines 641-649 in train script):
#   cell_types = filtered_cell_types  # e.g., ['fibroblast', 'macrophage', ...]
#   cell_type_codes = filtered_codes   # e.g., {'fibroblast': 0, 'macrophage': 1, ...}  (NEW SEQUENTIAL CODES!)
#   code_remapping = {...}             # e.g., {67: 0, 9: 1, ...} (ORIGINAL -> NEW)

# So in validation:
#   self.cell_type_codes = filtered_codes = {'fibroblast': 0, 'macrophage': 1, ...}
#   Line 308 does: cell_type_codes.get('fibroblast') = 0 (already the NEW filtered code!)
#   Then line 497 does: remap_tensor[0] = code_remapping[0] = ???

# The bug is: validation is using FILTERED codes, but code_remapping expects ORIGINAL codes!

# The fix should be: validation should use ORIGINAL cell_type_codes, not filtered ones

print("BUG IDENTIFIED:")
print("="*60)
print("When cell type filtering is enabled:")
print()
print("Training script does (line 648-649):")
print("  cell_types = filtered_cell_types")
print("  cell_type_codes = filtered_codes  # {'fibroblast': 0, ...}")
print()
print("But code_remapping expects (line 454-461):")
print("  code_remapping[ORIGINAL_code] = NEW_code")
print("  e.g., code_remapping[67] = 0  # 67 is original code for 'fibroblast'")
print()
print("Validation does (line 308):")
print("  labels = [cell_type_codes.get(ct, -1) for ct in cell_types]")
print("  # Uses filtered_codes, so 'fibroblast' -> 0 (already filtered!)")
print()  
print("Then (line 497):")
print("  y = remap_tensor[y]")
print("  # Tries to remap 0, but code_remapping[0] doesn't exist!")
print("  # code_remapping only has keys like 67, 9, etc. (original codes)")
print("="*60)
print()
print("Solution: validation should use ORIGINAL cell_type_codes,")
print("NOT the filtered codes!")


BUG IDENTIFIED:
When cell type filtering is enabled:

Training script does (line 648-649):
  cell_types = filtered_cell_types
  cell_type_codes = filtered_codes  # {'fibroblast': 0, ...}

But code_remapping expects (line 454-461):
  code_remapping[ORIGINAL_code] = NEW_code
  e.g., code_remapping[67] = 0  # 67 is original code for 'fibroblast'

Validation does (line 308):
  labels = [cell_type_codes.get(ct, -1) for ct in cell_types]
  # Uses filtered_codes, so 'fibroblast' -> 0 (already filtered!)

Then (line 497):
  y = remap_tensor[y]
  # Tries to remap 0, but code_remapping[0] doesn't exist!
  # code_remapping only has keys like 67, 9, etc. (original codes)

Solution: validation should use ORIGINAL cell_type_codes,
NOT the filtered codes!


In [17]:

# Let's look at an actual training run to see what code_remapping looks like
# Check if there's a saved config or checkpoint with this info

checkpoint_dir = Path('/localdata/checkpoints/mlp_composable_test')
if checkpoint_dir.exists():
    # Try to find the latest checkpoint
    checkpoints = sorted(checkpoint_dir.glob('*.pth'))
    if checkpoints:
        latest = checkpoints[-1]
        print(f"Loading checkpoint: {latest.name}")
        ckpt = torch.load(latest, weights_only=False)
        
        # Check if code_remapping is saved
        if 'code_remapping' in ckpt:
            code_remap = ckpt['code_remapping']
            print(f"\ncode_remapping found in checkpoint!")
            print(f"Number of mappings: {len(code_remap)}")
            print(f"\nFirst 20 mappings (original -> filtered):")
            for i, (orig, filt) in enumerate(list(code_remap.items())[:20]):
                print(f"  {orig:3d} -> {filt:3d}")
        else:
            print("\ncode_remapping not found in checkpoint")
            print(f"Available keys: {list(ckpt.keys())}")


In [18]:

# Try looking for WandB logs or training output that might show the remapping
import os

# Look for recent training logs
log_files = sorted(Path('/localdata/checkpoints/mlp_composable_test').glob('*.log'))
if log_files:
    print(f"Found {len(log_files)} log files")
    latest_log = log_files[-1]
    print(f"Latest: {latest_log.name}")
else:
    print("No log files found")
    
# Let's actually simulate what would happen with filtering
# by loading the cell counts file and seeing what gets filtered

cell_counts_file = Path('/localdata/training_data/cellxgene_v2_training_cell_counts.json')
if cell_counts_file.exists():
    import json
    with open(cell_counts_file, 'r') as f:
        cell_counts = json.load(f)
    
    print(f"\nTotal cell types: {len(cell_counts)}")
    print(f"Counts range: {min(cell_counts.values())} to {max(cell_counts.values())}")
    
    # Sort by count
    sorted_counts = sorted(cell_counts.items(), key=lambda x: x[1], reverse=True)
    print(f"\nTop 10 cell types by count:")
    for ct, count in sorted_counts[:10]:
        print(f"  {ct}: {count:,}")
    
    print(f"\nBottom 10 cell types by count:")
    for ct, count in sorted_counts[-10:]:
        print(f"  {ct}: {count:,}")


No log files found


In [19]:

# Let me create a simple test to understand the actual behavior
# by simulating the filtering process

# Assume we have original cell types with codes
original_cell_types = ['cell_A', 'cell_B', 'cell_C', 'cell_D', 'cell_E']
original_codes = pd.Series(range(len(original_cell_types)), index=original_cell_types)
print("Original cell_type_codes:")
print(original_codes)

# Simulate filtering: let's say we filter out cell_B and cell_D (low counts)
included_types = ['cell_A', 'cell_C', 'cell_E']
excluded_types = ['cell_B', 'cell_D']

# Create filtered codes (line 451 from train script)
filtered_codes = pd.Series(range(len(included_types)), index=included_types)
print("\nFiltered cell_type_codes (what gets passed to trainer):")
print(filtered_codes)

# Create code_remapping (lines 454-467 from train script)
code_remapping = {}
for cell_type in included_types:
    if cell_type in original_codes.index:
        original_code = original_codes[cell_type]
        new_code = filtered_codes[cell_type]
        code_remapping[original_code] = new_code

for cell_type in excluded_types:
    if cell_type in original_codes.index:
        original_code = original_codes[cell_type]
        code_remapping[original_code] = -100

print("\ncode_remapping (original -> filtered):")
for orig, filt in sorted(code_remapping.items()):
    print(f"  {orig} -> {filt}")

print("\n" + "="*60)
print("THE KEY INSIGHT:")
print("="*60)
print("Original codes: 0=A, 1=B, 2=C, 3=D, 4=E")
print("Filtered codes: 0=A, 1=C, 2=E  (B and D excluded)")
print("code_remapping: {0->0, 1->-100, 2->1, 3->-100, 4->2}")
print()
print("Notice: 2 -> 1 and 4 -> 2 (NOT monotonic!)")
print("The remapping RESEQUENCES the codes to be contiguous 0,1,2,...")


Original cell_type_codes:
cell_A    0
cell_B    1
cell_C    2
cell_D    3
cell_E    4
dtype: int64

Filtered cell_type_codes (what gets passed to trainer):
cell_A    0
cell_C    1
cell_E    2
dtype: int64

code_remapping (original -> filtered):
  0 -> 0
  1 -> -100
  2 -> 1
  3 -> -100
  4 -> 2

THE KEY INSIGHT:
Original codes: 0=A, 1=B, 2=C, 3=D, 4=E
Filtered codes: 0=A, 1=C, 2=E  (B and D excluded)
code_remapping: {0->0, 1->-100, 2->1, 3->-100, 4->2}

Notice: 2 -> 1 and 4 -> 2 (NOT monotonic!)
The remapping RESEQUENCES the codes to be contiguous 0,1,2,...


In [20]:

print("VALIDATION BUG DEMONSTRATION:")
print("="*60)

# In trainer.__init__, when filtering is enabled:
cell_types_passed = included_types  # ['cell_A', 'cell_C', 'cell_E']
cell_type_codes_passed = filtered_codes  # {A:0, C:1, E:2}
code_remapping_passed = code_remapping  # {0:0, 1:-100, 2:1, 3:-100, 4:2}

print("Trainer receives:")
print(f"  cell_types: {cell_types_passed}")
print(f"  cell_type_codes: {dict(cell_type_codes_passed)}")
print(f"  code_remapping: {code_remapping_passed}")

print("\nValidation dataset receives:")
print(f"  cell_type_codes: {dict(cell_type_codes_passed)}")
print(f"  code_remapping: {code_remapping_passed}")

print("\nValidation parquet has cell_type='cell_C':")
print("  Step 1 (line 308): labels = cell_type_codes.get('cell_C') = 1")
print("           ^ Uses filtered codes, so C -> 1")

print("\n  Step 2 (line 497): y = remap_tensor[y] = remap_tensor[1]")
print("           code_remapping[1] = -100 (because original code 1 was cell_B, excluded!)")
print("           ^ So 'cell_C' gets filtered out!")

print("\n  Expected: 'cell_C' should map to filtered code 1 ✓")
print("  Actual:   'cell_C' maps to 1, then 1 gets remapped to -100 ✗")

print("\n" + "="*60)
print("ROOT CAUSE: validation uses filtered_codes in step 1,")
print("but code_remapping expects original_codes!")
print("="*60)


VALIDATION BUG DEMONSTRATION:
Trainer receives:
  cell_types: ['cell_A', 'cell_C', 'cell_E']
  cell_type_codes: {'cell_A': np.int64(0), 'cell_C': np.int64(1), 'cell_E': np.int64(2)}
  code_remapping: {np.int64(0): np.int64(0), np.int64(2): np.int64(1), np.int64(4): np.int64(2), np.int64(1): -100, np.int64(3): -100}

Validation dataset receives:
  cell_type_codes: {'cell_A': np.int64(0), 'cell_C': np.int64(1), 'cell_E': np.int64(2)}
  code_remapping: {np.int64(0): np.int64(0), np.int64(2): np.int64(1), np.int64(4): np.int64(2), np.int64(1): -100, np.int64(3): -100}

Validation parquet has cell_type='cell_C':
  Step 1 (line 308): labels = cell_type_codes.get('cell_C') = 1
           ^ Uses filtered codes, so C -> 1

  Step 2 (line 497): y = remap_tensor[y] = remap_tensor[1]
           code_remapping[1] = -100 (because original code 1 was cell_B, excluded!)
           ^ So 'cell_C' gets filtered out!

  Expected: 'cell_C' should map to filtered code 1 ✓
  Actual:   'cell_C' maps to 1, the